<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/10%20-%20Avaliacao%20Modulo%201%20Motor%20de%20Intertravamento%20e%20Diagnostico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 10 - Notebook: Avaliação Integrada do Módulo 1 — SCADA-Core e Diagnóstico do AGV

Este notebook consolida a suíte completa de segurança e diagnóstico do AGV para o Módulo 1: Normalização de Sinais Mecatrônicos, Provador Dedutivo Formal por $2^n$ estados, e Motor Híbrido de Inferência (*Forward* e *Backward Chaining*).

---

### Célula 1 (Texto / Markdown)
```markdown
## 1. Mapeador e Normalizador de Sinais de Campo da Telemetria
Converte leituras brutas de sensores mecatrônicos (LiDAR, Encoders, BMS, Wi-Fi) para o espaço de proposições booleanas da Base de Fatos dinâmicos $\mathcal{F}(t)$.

In [1]:
import itertools
from dataclasses import dataclass
from typing import Set, Tuple, List, Dict, Any, Optional

class NormalizadorTelemetriaAGV:
    """Mapeia sinais físicos brutos do robô para proposições lógicas booleanas."""

    @staticmethod
    def processar_sinais(
        distancia_lidar_m: float,
        velocidade_mps: float,
        temp_bms_c: float,
        corrente_bms_a: float,
        wifi_conectado: bool,
        pwm_motor: int,
        rpm_encoder: int,
        soc_bateria_pct: float,
        carga_engatada: bool
    ) -> Dict[str, bool]:
        return {
            "LIDAR_ZONA_VERMELHA": distancia_lidar_m <= 0.5,
            "VELOCIDADE_GT_ZERO": velocidade_mps > 0.02,
            "BMS_TEMP_HIGH": temp_bms_c >= 60.0,
            "CORRENTE_CARGA_HIGH": corrente_bms_a >= 25.0,
            "FALHA_COMUNIC_WIFI": not wifi_conectado,
            "MOTOR_PWM_HIGH": pwm_motor >= 75,
            "ENCODER_RPM_ZERO": rpm_encoder == 0,
            "BATERIA_SOC_LOW": soc_bateria_pct <= 15.0,
            "CARGA_ENGATADA": carga_engatada
        }

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Utilitário para exibição de relatórios em ASCII puro."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

print("Módulo de Telemetria e Utilitários de Exibição compilados.")

Módulo de Telemetria e Utilitários de Exibição compilados.


## 2. Provador Dedutivo Formal (Validação por Tabela-Verdade $2^n$)
Verifica matematicamente se o algoritmo de intertravamento do AGV constitui uma Tautologia e realiza Prova por Refutação antes de enviar o sinal para o hardware de potência (Ponte H / Freio Eletromagnético).

In [2]:
class ProvadorDedutivoAGV:
    """Motor de validação dedutiva formal por tabela-verdade exaustiva."""
    def __init__(self, variaveis: List[str]):
        self.variaveis = variaveis
        self.espaco_estados = list(itertools.product([False, True], repeat=len(variaveis)))

    def validar_intertravamento(self, premissas: List[Any], conclusao: Any) -> Dict[str, Any]:
        argumento_valido = True
        refutado = True
        linhas_contraexemplo = []

        for estado in self.espaco_estados:
            ctx = dict(zip(self.variaveis, estado))
            p_vals = [p(**ctx) for p in premissas]
            todas_p = all(p_vals)
            c_val = conclusao(**ctx)

            # Teste de Validade Semântica (Se Premissas V => Conclusão DEVE ser V)
            if todas_p and not c_val:
                argumento_valido = False
                refutado = False
                linhas_contraexemplo.append(ctx)

        return {
            "valido": argumento_valido,
            "refutado_com_sucesso": refutado,
            "contraexemplos": linhas_contraexemplo
        }

# Validação do Intertravamento SIL 3 de Emergência do AGV
provador = ProvadorDedutivoAGV(["d1", "v1", "Trip"])

# Premissa 1: (d1 ∧ v1) => Trip
P1 = lambda d1, v1, Trip: not (d1 and v1) or Trip
# Premissa 2: d1 ∧ v1 (Invasão de zona e robô andando)
P2 = lambda d1, v1, Trip: d1 and v1
# Conclusão: Trip (Corte de Potência)
C = lambda d1, v1, Trip: Trip

resultado_prova = provador.validar_intertravamento([P1, P2], C)

print(f"--- PROVA FORMAL DE TAUTOLOGIA (INTERTRAVAMENTO SIL 3) ---")
print(f"Intertravamento Matematicamente VÁLIDO? -> {resultado_prova['valido']}")
print(f"Insatisfatibilidade da Refutação Confirmada? -> {resultado_prova['refutado_com_sucesso']}")
assert resultado_prova['valido'] == True

--- PROVA FORMAL DE TAUTOLOGIA (INTERTRAVAMENTO SIL 3) ---
Intertravamento Matematicamente VÁLIDO? -> True
Insatisfatibilidade da Refutação Confirmada? -> True


## 3. Motor Híbrido de Inferência em Lógica de Produção
Combina **Forward Chaining** (para disparo reativo instantâneo de alarmes) e **Backward Chaining** (para investigação auditável de causa-raiz no SCADA).

In [3]:
@dataclass
class RegraProducao:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    diagnostico: str
    prioridade: int
    tempo_resposta_ms: int

class MotorInferenciaHibridoAGV:
    def __init__(self):
        self.regras: List[RegraProducao] = []

    def adicionar_regra(self, id_r: str, antecede: List[str], conseq: str, diag: str, prio: int, t_ms: int):
        self.regras.append(RegraProducao(id_r, set(antecede), conseq, diag, prio, t_ms))

    def forward_chaining(self, fatos_campo: Set[str]) -> Tuple[Set[str], List[Dict[str, Any]]]:
        fatos_ativos = set(fatos_campo)
        trilha = []
        passo = 1

        while True:
            # Pattern Matching
            candidatas = [r for r in self.regras if r.antecedentes.issubset(fatos_ativos) and r.consequente not in fatos_ativos]
            if not candidatas:
                break

            # Resolução de Conflitos (Prioridade ISO 13849 e Tempo de Resposta)
            candidatas.sort(key=lambda r: (-r.prioridade, r.tempo_resposta_ms))
            regra_vencedora = candidatas[0]

            fatos_ativos.add(regra_vencedora.consequente)
            trilha.append({
                "Passo": passo,
                "Regra": regra_vencedora.id_regra,
                "Fato Inferido": regra_vencedora.consequente,
                "Causa-Raiz": regra_vencedora.diagnostico,
                "Prioridade": regra_vencedora.prioridade,
                "Latência": f"{regra_vencedora.tempo_resposta_ms} ms"
            })
            passo += 1

        return fatos_ativos, trilha

    def backward_chaining(self, meta: str, fatos_ativos: Set[str], trilha: Optional[List[Dict[str, Any]]] = None) -> Tuple[bool, List[Dict[str, Any]]]:
        if trilha is None:
            trilha = []

        if meta in fatos_ativos:
            trilha.append({"Meta/Submeta": meta, "Status": "CONFIRMADO (Telemetria/Fato)", "Regra": "N/A"})
            return True, trilha

        candidatas = [r for r in self.regras if r.consequente == meta]
        candidatas.sort(key=lambda r: -r.prioridade)

        for regra in candidatas:
            submetas_ok = True
            for ant in regra.antecedentes:
                ok, _ = self.backward_chaining(ant, fatos_ativos, trilha)
                if not ok:
                    submetas_ok = False
                    break

            if submetas_ok:
                fatos_ativos.add(meta)
                trilha.append({"Meta/Submeta": meta, "Status": "PROVADO VIA REGRA", "Regra": regra.id_regra})
                return True, trilha

        trilha.append({"Meta/Submeta": meta, "Status": "FALHA DE PROVA", "Regra": "N/A"})
        return False, trilha

print("Motor Inferencia Hibrido AGV inicializado.")

Motor Inferencia Hibrido AGV inicializado.


## 4. Suíte Integrada de Testes de Estresse e Validação 100% (Módulo 1)
Execução da pipeline completa: Telemetria Bruta $\rightarrow$ Normalização $\rightarrow$ Inferência Forward $\rightarrow$ Auditoria Backward.

In [4]:
# 1. Instanciação e carga das regras R-01 a R-06
motor_scada = MotorInferenciaHibridoAGV()

motor_scada.adicionar_regra("R-01", ["LIDAR_ZONA_VERMELHA", "VELOCIDADE_GT_ZERO"], "RISCO_COLISAO_IMINENTE", "Obstáculo Dinâmico em Rota", 10, 50)
motor_scada.adicionar_regra("R-02", ["RISCO_COLISAO_IMINENTE", "FALHA_COMUNIC_WIFI"], "TRIP_ISOLAMENTO_TOTAL", "AGV Cego e Incomunicável", 10, 10)
motor_scada.adicionar_regra("R-03", ["MOTOR_PWM_HIGH", "ENCODER_RPM_ZERO"], "ROTOR_BLOQUEADO", "Travamento Mecânico na Tração", 8, 100)
motor_scada.adicionar_regra("R-04", ["BMS_TEMP_HIGH", "CORRENTE_CARGA_HIGH"], "RISCO_FUGA_TERMICA", "Sobreaquecimento Severo de Bateria", 9, 150)
motor_scada.adicionar_regra("R-05", ["BATERIA_SOC_LOW", "CARGA_ENGATADA"], "BMS_DESCARGA_CRITICA", "Risco de Apagão com Carga", 4, 500)

# 2. Injeção de Sinal Mecatrônico do Cenário de Crítico de Campo
# Telemetria Bruta do AGV:
# - LiDAR = 0.3m (Zona Vermelha)
# - Vel = 0.5m/s (Em movimento)
# - Wi-Fi = Desconectado (False)
sinais_brutos_cenario = NormalizadorTelemetriaAGV.processar_sinais(
    distancia_lidar_m=0.3,
    velocidade_mps=0.5,
    temp_bms_c=35.0,
    corrente_bms_a=10.0,
    wifi_conectado=False,
    pwm_motor=90,
    rpm_encoder=0,
    soc_bateria_pct=80.0,
    carga_engatada=True
)

fatos_iniciais = {tag for tag, valor in sinais_brutos_cenario.items() if valor}

print("=========================================================")
print("   SISTEMA INTEGRADO SCADA-CORE AGV - MÓDULO 1 DE TESTE")
print("=========================================================\n")

print("1. Fatos Extraídos da Telemetria de Campo:")
for f in sorted(fatos_iniciais):
    print(f"  [+] {f}")

# 3. Execução Forward Chaining
fatos_finais, log_forward = motor_scada.forward_chaining(fatos_iniciais)

print("\n2. Trilha de Inferência Reativa (Forward Chaining):")
print(formatar_tabela(log_forward))

# 4. Execução Backward Chaining para Investigação de Causa-Raiz
meta_investigacao = "TRIP_ISOLAMENTO_TOTAL"
provado, log_backward = motor_scada.backward_chaining(meta_investigacao, fatos_iniciais.copy())

print(f"\n3. Investigação Auditoria SCADA para Meta '{meta_investigacao}':")
print(f"   Status da Causa-Raiz: {'CONFIRMADA' if provado else 'DESCARTADA'}\n")
print(formatar_tabela(log_backward))

# 5. Assertivas de Validação da Suíte
assert "TRIP_ISOLAMENTO_TOTAL" in fatos_finais
assert "ROTOR_BLOQUEADO" in fatos_finais
assert provado == True

print("\n=========================================================")
print("   SUÍTE DE TESTES DO MÓDULO 1: 100% APROVADA E VALIDADA")
print("=========================================================")

   SISTEMA INTEGRADO SCADA-CORE AGV - MÓDULO 1 DE TESTE

1. Fatos Extraídos da Telemetria de Campo:
  [+] CARGA_ENGATADA
  [+] ENCODER_RPM_ZERO
  [+] FALHA_COMUNIC_WIFI
  [+] LIDAR_ZONA_VERMELHA
  [+] MOTOR_PWM_HIGH
  [+] VELOCIDADE_GT_ZERO

2. Trilha de Inferência Reativa (Forward Chaining):
Passo | Regra | Fato Inferido          | Causa-Raiz                    | Prioridade | Latência
------+-------+------------------------+-------------------------------+------------+---------
1     | R-01  | RISCO_COLISAO_IMINENTE | Obstáculo Dinâmico em Rota    | 10         | 50 ms   
2     | R-02  | TRIP_ISOLAMENTO_TOTAL  | AGV Cego e Incomunicável      | 10         | 10 ms   
3     | R-03  | ROTOR_BLOQUEADO        | Travamento Mecânico na Tração | 8          | 100 ms  

3. Investigação Auditoria SCADA para Meta 'TRIP_ISOLAMENTO_TOTAL':
   Status da Causa-Raiz: CONFIRMADA

Meta/Submeta           | Status                       | Regra
-----------------------+------------------------------+------
LI